In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
prefix = ""


In [ ]:
%pip install unittest-xml-reporting

In [ ]:
import xmlrunner

In [ ]:
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

# Initialize Spark session
spark = SparkSession.builder \
    .appName("AI Enrichments TA4H Tests") \
    .getOrCreate()

class TextAnalyticsForHealthDataValidationTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id

    def setUp(self):
        self.token = mssparkutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
        self.runtime_context = mssparkutils.runtime.context
        print(self.runtime_context)

    def test_bronze_ai_enrichments_ingestion_folders_are_empty(self):

        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/AIEnrichments/Text/Fabric-HDS"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/AIEnrichments/Object/Fabric-HDS"))
        
    def test_ta4h_processed_text_data(self):

        processed_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/AIEnrichments/Text/Fabric-HDS/")
        self.assertEqual(1, len(processed_files_folder))

        year_folders = mssparkutils.fs.ls(processed_files_folder[0].path)
        self.assertEqual(1, len(year_folders))

        months_folder = mssparkutils.fs.ls(year_folders[0].path)
        self.assertEqual(1, len(months_folder))

        days_folder = mssparkutils.fs.ls(months_folder[0].path)
        self.assertEqual(1, len(days_folder))

        
        enrichment_files = mssparkutils.fs.ls(days_folder[0].path)
        ndjson_files = [file for file in enrichment_files if file.name.endswith(".ndjson")]    
        self.assertEqual(1, len(ndjson_files), "Expected exactly one NDJSON file under the day folder for text enrichments.")

    def test_ta4h_processed_object_data(self):

        processed_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/AIEnrichments/Object/Fabric-HDS/")
        self.assertEqual(1, len(processed_files_folder))

        year_folders = mssparkutils.fs.ls(processed_files_folder[0].path)
        self.assertEqual(1, len(year_folders))

        months_folder = mssparkutils.fs.ls(year_folders[0].path)
        self.assertEqual(1, len(months_folder))

        days_folder = mssparkutils.fs.ls(months_folder[0].path)
        self.assertEqual(1, len(days_folder))

        enrichment_files = mssparkutils.fs.ls(days_folder[0].path)
        ndjson_files = [file for file in enrichment_files if file.name.endswith(".ndjson")]
        self.assertEqual(1, len(ndjson_files), "Expected one NDJSON file under the day folder for object enrichments.")

    def test_ai_enrichments_bronze_table_is_populated(self):

        df = self.spark.sql("SELECT type, COUNT(*) as count FROM AIEnrichments GROUP BY type")
        
        results = df.collect()
        resource_counts = {row['type']: row['count'] for row in results}

        self.assertGreater(resource_counts.get("text", 0), 0)
        self.assertGreater(resource_counts.get("object", 0), 0)

    def test_ai_enrichments_silver_text_table_is_populated(self):

        df = self.spark.sql(f"SELECT COUNT(*) as count FROM {prefix}_msft_silver.TextEnrichments")
        
        results = df.collect()        
        record_count = results[0]['count']

        self.assertGreater(record_count, 0, f"Expected record count to be more than 0.")

    def test_ai_enrichments_silver_object_table_is_populated(self):

        df = self.spark.sql(f"SELECT COUNT(*) as count FROM {prefix}_msft_silver.ObjectEnrichments")
        
        results = df.collect()        
        record_count = results[0]['count']

        self.assertGreater(record_count, 0, f"Expected record count to be more than 0.")

def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TextAnalyticsForHealthDataValidationTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)